# Generate Synthetic Credit Risk Assessment Data

## Business Scenario

A bank is building a Credit Risk Assessment solution.

The development team needs realistic loan application data for testing, reporting, and analytics. Using real customer information is not allowed because of privacy and compliance requirements.

To solve this problem, you'll use GPT-5.4-mini to generate synthetic credit application records that resemble real-world loan applications while containing no actual customer data.

## What You'll Build

In this lab, you'll create a synthetic dataset containing credit applications and lending decisions.

The generated records will include:

- Customer Information
- Employment Information
- Income Information
- Credit Information
- Loan Information
- Risk Assessment Results
- Lending Decisions

## Prerequisites

Before starting, ensure you have:

- Azure AI Foundry Project Endpoint
- Azure AI Foundry API Key
- GPT-5.4-mini Deployment
- Python 3.12 or above
- Basic Python knowledge

Estimated Time: 25 Minutes

## What You'll Learn

By the end of this lab, you'll be able to:

- Connect to GPT-5.4-mini
- Create structured schemas using Pydantic
- Generate synthetic business data
- Save AI-generated data
- Analyse datasets using Pandas
- Export data for further use

## What is Synthetic Data

Synthetic data is artificially generated information created by AI models that mimics real-world data patterns without containing actual personal information. One-liner: AI-generated fake data that looks real but protects privacy.

## Differences: Synthetic vs Real Data

| Aspect | Synthetic Data | Real Data |
|--------|---|---|
| Origin | Generated by AI model | Collected from actual sources |
| Privacy | No personal information | Contains real customer details |
| Legal Risk | Low or none | High (GDPR, CCPA compliance) |
| Consistency | Uniform patterns | Natural variations |
| Compliance | No regulations needed | Requires strict compliance |
| Cost | Cheap to generate | Expensive to collect and store |
| Testing | Perfect for development | Cannot use in testing safely |
| Relationships | May be unrealistic | True correlations present |
| Use Case | Testing and demos | Production systems |
| Biases | May not reflect reality | Real-world biases present |

## Step 1: Install Required Packages

Install the libraries needed for this lab: OpenAI client, Pydantic for schemas, and Pandas for data analysis.

> **Tip:** If you encounter an error during the execution of any cell, try the following steps in order:
>
> 1. Click **Clear All Outputs**.
> 2. Click **Restart** → **Restart Kernel**.
> 3. Go to **Step 3** in the notebook.
> 4. Replace `API_KEY` with your valid `gpt-5.4-mini` key from Azure.
> 5. Ensure that you have not modified any other part of the code.
> 6. Select **Run All**.
>
> This should resolve most common issues. If you still face an issue, please let us know and we’ll be happy to help resolve it.


In [ ]:
%pip install -qU openai pydantic pandas openpyxl

## Step 2: Verify Package Installation

Check that all packages are installed with correct versions.

In [ ]:
import openai
import pydantic
import pandas as pd

print("OpenAI:", openai.__version__)
print("Pydantic:", pydantic.__version__)
print("Pandas:", pd.__version__)

## Step 3: Configure GPT-5.4-mini

Replace the placeholders below with the values provided by your instructor. Get PROJECT_ENDPOINT and API_KEY from Azure AI Foundry.

In [ ]:
from openai import OpenAI

PROJECT_ENDPOINT = "https://hakunamatata1.services.ai.azure.com/openai/v1"
API_KEY = "replace_with_your_key"
MODEL_NAME = "gpt-5.4-mini"

client = OpenAI(
    base_url=PROJECT_ENDPOINT,
    api_key=API_KEY
)

print("Client initialized successfully.")

## Step 4: Create a Sample Credit Application

This sample record acts as the template that GPT-5.4-mini will follow when generating new applications.

In [ ]:
sample_record = {
    "application_id": "APP-1001",
    "customer_name": "John Smith",
    "age": 35,
    "employment_status": "Full-Time",
    "annual_income": 85000,
    "credit_score": 720,
    "loan_amount": 250000,
    "loan_purpose": "Home Purchase",
    "existing_debt": 15000,
    "debt_to_income_ratio": 0.18,
    "risk_category": "Low Risk",
    "decision": "Approved",
    "decision_reason": "Stable income and strong credit history"
}

## Step 5: Review the Sample

Display the sample record in a readable JSON format. Take a moment to review the structure before generating additional records.

In [ ]:
import json

print(json.dumps(sample_record, indent=2))

## Step 6: Define the Schema

Create Pydantic models to enforce data structure and type validation for credit applications and the wrapper container.

In [ ]:
from pydantic import BaseModel
from typing import List

class CreditApplication(BaseModel):
    application_id: str
    customer_name: str
    age: int
    employment_status: str
    annual_income: float
    credit_score: int
    loan_amount: float
    loan_purpose: str
    existing_debt: float
    debt_to_income_ratio: float
    risk_category: str
    decision: str
    decision_reason: str

class CreditApplicationWrapper(BaseModel):
    records: List[CreditApplication]

print("Schemas defined successfully.")

## Step 7: Configure the Number of Records

Set the number of credit applications to generate. You can change this value based on your testing needs. Start with 10 for testing.

In [ ]:
number_of_records = 10

print(f"Will generate {number_of_records} credit applications.")

## Step 8: Create the Prompt

Build the prompt that instructs GPT-5.4-mini how to generate credit applications. The prompt includes the sample record as a template for consistency.

In [ ]:
prompt = f"""You are helping a bank generate synthetic credit application data.

Generate exactly {number_of_records} credit applications.

Requirements:

- Use realistic customer names
- Use realistic employment information
- Use realistic annual income values
- Use realistic credit scores
- Use realistic debt amounts
- Use realistic loan amounts
- Include Low Risk, Medium Risk and High Risk customers
- Include realistic lending decisions
- Follow the sample structure exactly

Sample Record:

{json.dumps(sample_record, indent=2)}

Return JSON only.

Expected Structure:

{{
  "records": [
  ]
}}
"""

print("Prompt created successfully.")

## Step 9: Generate Synthetic Data

Call GPT-5.4-mini to generate synthetic credit application data based on the prompt. The model will create new records following the sample structure.

In [ ]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": prompt}]
)

generated_text = response.choices[0].message.content
print("Synthetic data generated successfully.")
print(f"Tokens used: {response.usage.total_tokens}")

## Step 10: Save Generated Data

Save the generated data to a JSON file. The helper function ensures files are not overwritten by adding version numbers.

In [ ]:
from pathlib import Path

def get_unique_filename(filename):
    file_path = Path(filename)
    if not file_path.exists():
        return filename
    counter = 1
    while True:
        new_name = f"{file_path.stem}_{counter}{file_path.suffix}"
        if not Path(new_name).exists():
            return new_name
        counter += 1

json_file = get_unique_filename("synthetic_credit_applications.json")
with open(json_file, "w") as f:
    f.write(generated_text)

print(f"Saved: {json_file}")

## Step 11: Load Generated Data

Read the JSON file and load it into memory. If files already exist, auto-versioning prevents overwriting.

In [ ]:
with open(json_file, "r") as f:
    generated_data = json.load(f)

print(f"Applications Generated: {len(generated_data['records'])}")

## Step 12: Convert Data to Pandas

Transform the nested JSON structure into a Pandas DataFrame for easier analysis and manipulation.

In [ ]:
df = pd.json_normalize(generated_data["records"])
print(df.head())

## Step 13: Perform Basic Analysis

Calculate key statistics from the generated credit applications.

In [ ]:
print(f"Total Applications: {len(df)}")
print(f"Average Credit Score: {round(df['credit_score'].mean(), 2)}")
print(f"Average Loan Amount: ${round(df['loan_amount'].mean(), 2)}")
print(f"Average Annual Income: ${round(df['annual_income'].mean(), 2)}")

## Step 14: Analyze Risk Distribution

Count the number of applications in each risk category to verify the distribution.

In [ ]:
print("Risk Distribution:")
print(df["risk_category"].value_counts())

## Step 15: Analyze Lending Decisions

Count the number of Approved, Denied, and Pending decisions in the generated dataset.

In [ ]:
print("Lending Decisions:")
print(df["decision"].value_counts())

## Step 16: Data Quality Checks

Verify data integrity by checking for missing values and duplicate records.

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print(f"\nDuplicate Records: {df.duplicated().sum()}")

## Step 17: Export to CSV

Save the DataFrame to a CSV file for use in spreadsheet applications or data analysis tools.

In [ ]:
csv_file = get_unique_filename("synthetic_credit_applications.csv")
df.to_csv(csv_file, index=False)
print(f"Saved: {csv_file}")

## Step 18: Export to Excel

Save the DataFrame to an Excel file with formatted columns for better readability and professional presentation.

In [ ]:
excel_file = get_unique_filename("synthetic_credit_applications.xlsx")
df.to_excel(excel_file, index=False, sheet_name="Credit Applications")
print(f"Saved: {excel_file}")

## Step 19: Verify All Output Files

List all generated files to confirm successful exports.

In [ ]:
print("Generated Files:")
print(f"- {json_file}")
print(f"- {csv_file}")
print(f"- {excel_file}")

## Expected Output Files

After completing the lab, you should see files similar to:

- synthetic_credit_applications.json
- synthetic_credit_applications.csv
- synthetic_credit_applications.xlsx

If previous files already exist, auto-versioning creates:

- synthetic_credit_applications_1.json
- synthetic_credit_applications_1.csv
- synthetic_credit_applications_1.xlsx

### Tips for Real-World Use

- Use realistic sample records for better results
- Start with 5 to 10 records while testing
- Review generated applications before using them
- Save your work frequently
- Compare risk categories with lending decisions
- Use Pandas filtering to explore the generated data
- Generate new datasets only when needed

## What You Learned

In this lab, you learned how to:

- Connect to GPT-5.4-mini
- Generate synthetic credit application datasets
- Create structured schemas using Pydantic
- Save AI-generated data to multiple formats
- Analyse lending decisions using Pandas
- Export business datasets for further use
- Work with a real-world credit risk assessment use case

## Conclusion

In this lab, you generated synthetic credit application data using GPT-5.4-mini and analysed the results using Pandas.

The same approach can be applied to:

- Credit Risk Assessment
- Loan Processing
- Fraud Detection
- Customer Onboarding
- Insurance Claims
- Financial Analytics
- Testing and Demo Environments

You now have a repeatable process for generating realistic synthetic business data without using sensitive customer information.